**What we compare:**

Model A is a small CNN trained from scratch.
Model B is a pretrained ResNet18 (transfer learning).
Model C is the same ResNet18 with attention pooling over slices instead of mean pooling.

All three train on the soft labels from the 4,349 non-gold studies and are scored on the 58 gold studies.

# **1: Load cache and labels**

load all 7.8 GB of images into RAM, align them with the soft labels, and split. 4,349 non-gold studies go to training (90%) and validation (10%). The 58 gold studies are never used for training or model selection.

In [1]:
import os, glob, time, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda"
ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
def find(pattern):
    hits = glob.glob(pattern, recursive=True); assert hits, f"not found: {pattern}"; return hits[0]

CIDX = find("/kaggle/input/datasets/*/rsna-knee-cache-192/**/cache_index.csv")
CD = os.path.dirname(CIDX)
soft = pd.read_csv(find("/kaggle/input/datasets/*/rsna-knee-soft-labels/soft_labels_final.csv"))
idx_df = pd.read_csv(CIDX)
assert (idx_df.StudyInstanceUID.values == soft.StudyInstanceUID.values).all(), "cache and labels not aligned"
N, K, S = len(idx_df), 16, 192
X = np.empty((N, 3, K, S, S), np.uint8); p = 0; t0 = time.time()
for f in sorted(glob.glob(f"{CD}/shard_*.npy")):
    a = np.load(f); X[p:p + len(a)] = a; p += len(a)
assert p == N
print(f"loaded {X.shape} ({X.nbytes / 1e9:.1f} GB) in {time.time() - t0:.0f}s")

train = pd.read_csv(f"{ROOT}/train.csv", usecols=["StudyInstanceUID"] + LABELS)
assert (train.StudyInstanceUID.values == soft.StudyInstanceUID.values).all()
is_gold = soft.is_gold.values.astype(bool)
gold_idx, nongold = np.where(is_gold)[0], np.where(~is_gold)[0]
T = soft[LABELS].values.astype(np.float32)                  # soft training targets (0 to 1)
Yg = train.loc[gold_idx, LABELS].astype(int).values         # real labels, evaluation only
perm = np.random.RandomState(SEED).permutation(nongold); nv = len(perm) // 10
va, tr = np.sort(perm[:nv]), np.sort(perm[nv:])
print("train:", len(tr), "| val:", len(va), "| gold:", len(gold_idx))

loaded (4407, 3, 16, 192, 192) (7.8 GB) in 54s
train: 3915 | val: 434 | gold: 58


# **2: Model definitions**
Concept: one backbone turns each slice into a feature vector. The slices of a plane are pooled (mean or attention) into one vector, then the 3 planes are joined into a single 12-output classifier. We don't flip images, because a flip would swap medial and lateral.

In [2]:
import timm
class KneeNet(nn.Module):
    def __init__(self, backbone, d, pool="mean"):
        super().__init__()
        self.bb, self.pool = backbone, pool
        self.att = nn.Linear(d, 1) if pool == "attn" else None
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(3 * d, 12))
    def forward(self, x):                                   # x: (B, 3 planes, slices, H, W)
        B, P, Ks, H, W = x.shape
        f = self.bb(x.reshape(B * P * Ks, 1, H, W)).view(B, P, Ks, -1)
        if self.pool == "mean":
            g = f.mean(2)
        else:
            w = torch.softmax(self.att(f).squeeze(-1), dim=2)
            g = (w.unsqueeze(-1) * f).sum(2)
        return self.head(g.reshape(B, -1))

def make_cnn():
    def blk(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2))
    return nn.Sequential(blk(1, 16), blk(16, 32), blk(32, 64), blk(64, 128), blk(128, 128),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten()), 128

def make_resnet():
    return timm.create_model("resnet18", pretrained=True, in_chans=1, num_classes=0), 512

# **3: Training and evaluation functions**
Concept: training uses 8 random slices per plane for speed and variety. Evaluation uses all 16. The best epoch is chosen by validation loss, never by gold. Gold is only read at the end.

In [3]:
KT = 8
def prep(xnp, train):
    x = torch.from_numpy(xnp).to(dev)                       # uint8 (B, 3, 16, H, W)
    if train:
        sel = torch.tensor(np.sort(np.random.choice(K, KT, replace=False)), device=dev)
        x = x.index_select(2, sel)
    x = x.float().div_(255)
    if train:
        B = x.shape[0]
        g = 1 + 0.2 * (torch.rand(B, 3, 1, 1, 1, device=dev) - 0.5)
        b = 0.1 * (torch.rand(B, 3, 1, 1, 1, device=dev) - 0.5)
        x = x * g + b
        dx, dy = np.random.randint(-12, 13, 2)
        x = torch.roll(x, shifts=(int(dy), int(dx)), dims=(3, 4))
    return (x - 0.45) / 0.225

@torch.no_grad()
def predict(model, idx, bs=8):
    model.eval(); out = []
    for s in range(0, len(idx), bs):
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.sigmoid(model(prep(X[idx[s:s + bs]], False)).float()).cpu().numpy())
    return np.concatenate(out)

def macro_auc(y, p):
    return float(np.mean([roc_auc_score(y[:, j], p[:, j]) for j in range(12) if len(set(y[:, j])) == 2]))

def boot_ci(y, p, n=1000, seed=0):
    rng = np.random.RandomState(seed); vals = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        vals.append(np.mean([roc_auc_score(y[i, j], p[i, j]) for j in range(12) if len(set(y[i, j])) == 2]))
    return np.percentile(vals, [2.5, 97.5])

PRED, RES = {}, {}
def run(name, make_backbone, pool, epochs, lr, bs=8):
    torch.manual_seed(SEED); np.random.seed(SEED)
    bb, d = make_backbone()
    model = KneeNet(bb, d, pool).to(dev)
    pr = T[tr].mean(0).clip(0.02, 0.98)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(np.clip((1 - pr) / pr, 1, 4), dtype=torch.float32, device=dev))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * int(np.ceil(len(tr) / bs))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda"); best = 1e9
    for ep in range(epochs):
        model.train(); order = np.random.permutation(tr); tot, t0 = 0.0, time.time()
        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            xb, yb = prep(X[idx], True), torch.tensor(T[idx], device=dev)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(xb)
            loss = loss_fn(out.float(), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
            tot += loss.item() * len(idx)
        pv = np.clip(predict(model, va), 1e-6, 1 - 1e-6); tv = T[va]
        vloss = float(-(tv * np.log(pv) + (1 - tv) * np.log(1 - pv)).mean())
        vauc = macro_auc((tv >= 0.5).astype(int), pv)
        print(f"{name} | epoch {ep + 1}/{epochs} | train loss {tot / len(order):.4f} | val loss {vloss:.4f} | val AUC vs labels {vauc:.3f} | {time.time() - t0:.0f}s", flush=True)
        if vloss < best:
            best = vloss; torch.save(model.state_dict(), f"/kaggle/working/{name}.pt")
    model.load_state_dict(torch.load(f"/kaggle/working/{name}.pt"))
    pg = predict(model, gold_idx); PRED[name] = pg
    lo, hi = boot_ci(Yg, pg)
    RES[name] = dict(model=name, gold_macro_AUC=round(macro_auc(Yg, pg), 3), CI_low=round(lo, 3), CI_high=round(hi, 3))
    print("GOLD:", RES[name])
    del model, opt; torch.cuda.empty_cache()

# **4: Model A, small CNN from scratch**
the "no pretraining" baseline. Run it first to check that the whole pipeline works.

In [4]:
run("cnn_scratch_mean", make_cnn, "mean", epochs=10, lr=1e-3)

cnn_scratch_mean | epoch 1/10 | train loss 0.9394 | val loss 0.6315 | val AUC vs labels 0.646 | 49s
cnn_scratch_mean | epoch 2/10 | train loss 0.9264 | val loss 0.5280 | val AUC vs labels 0.636 | 50s
cnn_scratch_mean | epoch 3/10 | train loss 0.9136 | val loss 0.6459 | val AUC vs labels 0.687 | 54s
cnn_scratch_mean | epoch 4/10 | train loss 0.8994 | val loss 0.6115 | val AUC vs labels 0.678 | 54s
cnn_scratch_mean | epoch 5/10 | train loss 0.8921 | val loss 0.5248 | val AUC vs labels 0.707 | 54s
cnn_scratch_mean | epoch 6/10 | train loss 0.8823 | val loss 0.5545 | val AUC vs labels 0.705 | 54s
cnn_scratch_mean | epoch 7/10 | train loss 0.8735 | val loss 0.5872 | val AUC vs labels 0.712 | 54s
cnn_scratch_mean | epoch 8/10 | train loss 0.8652 | val loss 0.5890 | val AUC vs labels 0.725 | 54s
cnn_scratch_mean | epoch 9/10 | train loss 0.8575 | val loss 0.5742 | val AUC vs labels 0.725 | 54s
cnn_scratch_mean | epoch 10/10 | train loss 0.8528 | val loss 0.5713 | val AUC vs labels 0.725 | 54s

# **5: Model B, pretrained ResNet18 with mean pooling**

transfer learning. ImageNet weights are adapted to grayscale and fine-tuned.

In [5]:
run("resnet18_mean", make_resnet, "mean", epochs=6, lr=2e-4)

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

resnet18_mean | epoch 1/6 | train loss 0.9288 | val loss 0.7939 | val AUC vs labels 0.709 | 125s
resnet18_mean | epoch 2/6 | train loss 0.8497 | val loss 0.5252 | val AUC vs labels 0.776 | 122s
resnet18_mean | epoch 3/6 | train loss 0.8090 | val loss 0.5007 | val AUC vs labels 0.795 | 122s
resnet18_mean | epoch 4/6 | train loss 0.7818 | val loss 0.5063 | val AUC vs labels 0.807 | 122s
resnet18_mean | epoch 5/6 | train loss 0.7623 | val loss 0.5156 | val AUC vs labels 0.811 | 122s
resnet18_mean | epoch 6/6 | train loss 0.7537 | val loss 0.5258 | val AUC vs labels 0.813 | 122s
GOLD: {'model': 'resnet18_mean', 'gold_macro_AUC': 0.672, 'CI_low': np.float64(0.614), 'CI_high': np.float64(0.725)}


# **6: Model C, ResNet18 with attention pooling**

 the model learns which slices matter, a practical use of attention .

In [6]:
run("resnet18_attn", make_resnet, "attn", epochs=6, lr=2e-4)

resnet18_attn | epoch 1/6 | train loss 0.9321 | val loss 0.8003 | val AUC vs labels 0.696 | 123s
resnet18_attn | epoch 2/6 | train loss 0.8547 | val loss 0.5070 | val AUC vs labels 0.764 | 122s
resnet18_attn | epoch 3/6 | train loss 0.8127 | val loss 0.5037 | val AUC vs labels 0.778 | 122s
resnet18_attn | epoch 4/6 | train loss 0.7826 | val loss 0.5158 | val AUC vs labels 0.789 | 122s
resnet18_attn | epoch 5/6 | train loss 0.7625 | val loss 0.5168 | val AUC vs labels 0.795 | 122s
resnet18_attn | epoch 6/6 | train loss 0.7545 | val loss 0.5356 | val AUC vs labels 0.796 | 122s
GOLD: {'model': 'resnet18_attn', 'gold_macro_AUC': 0.692, 'CI_low': np.float64(0.642), 'CI_high': np.float64(0.742)}


# **7: Comparison for the report**

macro AUC on the 58 gold studies with a 95% bootstrap interval, plus per-finding AUC.

In [7]:
res = pd.DataFrame(RES.values()).set_index("model"); print(res.to_string())
pf = pd.DataFrame({m: [round(roc_auc_score(Yg[:, j], P[:, j]), 2) for j in range(12)] for m, P in PRED.items()}, index=LABELS)
print("\nper-finding AUC on gold:\n", pf.to_string())
res.to_csv("/kaggle/working/image_model_results.csv"); pf.to_csv("/kaggle/working/image_model_per_finding_auc.csv")

                  gold_macro_AUC  CI_low  CI_high
model                                            
cnn_scratch_mean           0.631   0.568    0.690
resnet18_mean              0.672   0.614    0.725
resnet18_attn              0.692   0.642    0.742

per-finding AUC on gold:
                   cnn_scratch_mean  resnet18_mean  resnet18_attn
ACL                           0.63           0.67           0.69
MCL                           0.55           0.63           0.63
Medial Meniscus               0.61           0.56           0.60
Lateral Meniscus              0.66           0.71           0.71
Medial OA                     0.66           0.73           0.76
Lateral OA                    0.79           0.83           0.85
PF OA                         0.57           0.70           0.66
Effusion                      0.72           0.64           0.70
Synovitis                     0.63           0.57           0.61
Baker's                       0.72           0.79           0.78
Contusio

Results are in, and the pipeline works end to end.

| Model | Gold macro AUC | 95% interval |
|---|---|---|
| Small CNN from scratch | 0.631 | 0.568 to 0.690 |
| ResNet18, mean pooling | 0.672 | 0.614 to 0.725 |
| ResNet18, attention pooling | 0.692 | 0.642 to 0.742 |

**What it shows**
- **Transfer learning helps.** Both pretrained models beat the scratch CNN on gold (0.67 and 0.69 vs 0.63) and on the validation studies (0.81 and 0.80 vs 0.73).
- **Attention vs mean pooling is a tie.** Attention wins on gold (0.692 vs 0.672), but mean wins on validation (0.813 vs 0.796), and the gold intervals overlap heavily. With 58 studies we can't call a winner, and we should say so in the report.
- **Strongest findings:** Lateral OA (0.85), Baker's cyst (0.78), Medial OA (0.76). **Weakest:** Medial Meniscus (0.60), Synovitis (0.61), MCL and Fracture (0.63). Per-finding numbers on 58 studies are noisy, so treat them as hints.
- **One checkpoint detail:** we saved the epoch with the lowest validation loss, which was epoch 3. Validation AUC kept rising to epoch 6. Selecting by AUC could add a little. We'll try that later, after the first real submission.



# **8: ensemble check**
averaging two different models' predictions often beats either one.

In [8]:
Pm, Pa, Pc = PRED["resnet18_mean"], PRED["resnet18_attn"], PRED["cnn_scratch_mean"]
for name, P in [("mean+attn", (Pm + Pa) / 2), ("all three", (Pm + Pa + Pc) / 3)]:
    lo, hi = boot_ci(Yg, P); print(name, round(macro_auc(Yg, P), 3), (round(lo, 3), round(hi, 3)))

mean+attn 0.685 (np.float64(0.631), np.float64(0.738))
all three 0.684 (np.float64(0.635), np.float64(0.734))
